# Section 1: CVE Discovery Analysis — The Capability-Triggered Model

**ITESO | Vulnerability Management and the AI Crossroads**

This notebook queries the **NVD (National Vulnerability Database) API** directly to fetch real annual CVE counts, then uses statistical analysis to identify the structural break where AI-era discovery diverged from the historical trend.

**What we'll do:**
1. Fetch real CVE counts per year from the NVD API
2. Compute year-over-year change rates and CAGR
3. Fit a linear regression on the pre-AI baseline and project it forward
4. Quantify the structural break

**API used:** `https://services.nvd.nist.gov/rest/json/cves/2.0` (public, no key required for demo)

In [ ]:
import requests
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

OUTPUT = Path('output')
OUTPUT.mkdir(exist_ok=True)
CACHE = OUTPUT / 'nvd_yearly_counts.json'

NVD_API = 'https://services.nvd.nist.gov/rest/json/cves/2.0'
# Optional: set your NVD API key here for faster queries (50 req/30s vs 5 req/30s)
# Get a free key at https://nvd.nist.gov/developers/request-an-api-key
NVD_API_KEY = None  # e.g. 'your-key-here'

print('Setup complete.')

## 1. Fetching Real CVE Counts from NVD

The NVD API lets us count CVEs published in any date range using a single request — we only need `totalResults`, so we set `resultsPerPage=1` to minimise data transfer.

Results are cached so subsequent runs are instant. Delete `output/nvd_yearly_counts.json` to re-fetch.

In [ ]:
def fetch_nvd_year_count(year: int, api_key: str = None) -> int:
    """Return total CVEs published in a calendar year via NVD API."""
    params = {
        'pubStartDate': f'{year}-01-01T00:00:00.000',
        'pubEndDate':   f'{year}-12-31T23:59:59.999',
        'resultsPerPage': 1,
    }
    headers = {'apiKey': api_key} if api_key else {}
    r = requests.get(NVD_API, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    return r.json()['totalResults']


YEARS = list(range(2016, 2025))
delay = 3 if NVD_API_KEY else 7  # NVD rate limits: 50/30s with key, 5/30s without

if CACHE.exists():
    with open(CACHE) as f:
        counts = {int(k): v for k, v in json.load(f).items()}
    print(f'Loaded cached data for {sorted(counts.keys())}')
else:
    print(f'Fetching NVD counts for {YEARS}...')
    print(f'(~{len(YEARS) * delay}s without API key — set NVD_API_KEY above to go faster)')
    counts = {}
    for year in YEARS:
        try:
            n = fetch_nvd_year_count(year, NVD_API_KEY)
            counts[year] = n
            print(f'  {year}: {n:,}')
        except Exception as e:
            print(f'  {year}: FAILED ({e}) — skipping')
        time.sleep(delay)
    with open(CACHE, 'w') as f:
        json.dump(counts, f, indent=2)
    print(f'\nCached to {CACHE}')

df = pd.DataFrame({'year': list(counts.keys()), 'cves': list(counts.values())}).sort_values('year')
df

## 2. Year-over-Year Growth Rates and CAGR

Let's calculate the actual annual growth rates — and see if the commonly-repeated "25% per year" figure holds up.

In [ ]:
df = df.copy()
df['yoy_pct'] = df['cves'].pct_change() * 100

# CAGR over the pre-AI baseline (2016-2022)
pre_ai = df[df['year'] <= 2022].copy()
y0, yn = pre_ai['cves'].iloc[0], pre_ai['cves'].iloc[-1]
n_years = len(pre_ai) - 1
cagr = ((yn / y0) ** (1 / n_years) - 1) * 100

print('Year-over-Year Growth Rates:')
print(df[['year', 'cves', 'yoy_pct']].to_string(index=False,
      formatters={'cves': '{:,.0f}'.format, 'yoy_pct': lambda x: f'{x:+.1f}%' if pd.notna(x) else ''}))
print(f'\nPre-AI CAGR (2016–2022): {cagr:.1f}%')
print(f'Average annual YoY change (2017–2022): {df[df["year"].between(2017, 2022)]["yoy_pct"].mean():.1f}%')
print(f'\n2022→2023 jump: {df[df["year"]==2023]["yoy_pct"].values[0]:+.1f}%')
if 2024 in counts:
    print(f'2023→2024 jump: {df[df["year"]==2024]["yoy_pct"].values[0]:+.1f}%')

## 3. Linear Regression on the Pre-AI Baseline

We fit a linear regression on 2016–2022 data and project it forward. The gap between the trendline and actual counts is the **AI-driven excess** — the quantity we can no longer forecast with a simple time-series model.

In [ ]:
pre_ai_df = df[df['year'] <= 2022]
slope, intercept, r, p, se = stats.linregress(pre_ai_df['year'], pre_ai_df['cves'])

all_years = np.array(sorted(counts.keys()) + [2025, 2026])
trend_line = slope * all_years + intercept

# 95% confidence interval for the trendline
n = len(pre_ai_df)
mean_x = pre_ai_df['year'].mean()
ss_x = ((pre_ai_df['year'] - mean_x) ** 2).sum()
t_crit = stats.t.ppf(0.975, df=n - 2)
ci_margin = t_crit * se * np.sqrt(1 + 1/n + (all_years - mean_x)**2 / ss_x)

# Projections for 2025-2026 (illustrative estimates)
actual_with_proj = dict(counts)
actual_with_proj.update({2025: 55_000, 2026: 82_000})  # based on Q1 2026 rates

print(f'Regression fit (2016–2022):')
print(f'  Slope:     +{slope:,.0f} CVEs/year')
print(f'  R²:        {r**2:.3f}')
print(f'  Trend predicts 2024: {int(slope * 2024 + intercept):,}')
print(f'  Actual 2024:         {counts.get(2024, "(not yet fetched)"):,}')

## 4. Visualising the Structural Break

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.subplots_adjust(top=0.88, bottom=0.12)

# --- Left: absolute counts ---
ax = axes[0]
verified_years  = [y for y in all_years if y in counts]
projected_years = [y for y in all_years if y not in counts]

ax.bar(verified_years,  [counts[y] for y in verified_years],
       color='#4a90d9', alpha=0.85, label='Verified NVD/MITRE count')
if projected_years:
    ax.bar(projected_years, [actual_with_proj[y] for y in projected_years],
           color='#e06c5a', alpha=0.85, label='Estimate / projection')

ax.plot(all_years, trend_line, 'k--', lw=1.8, label=f'Pre-AI trend (2016–2022 fit)')
ax.fill_between(all_years, trend_line - ci_margin, trend_line + ci_margin,
                alpha=0.12, color='black', label='95% CI')

ax.fill_between(all_years,
                trend_line,
                [actual_with_proj.get(y, trend_line[i]) for i, y in enumerate(all_years)],
                where=[actual_with_proj.get(y, 0) > (slope * y + intercept) for y in all_years],
                alpha=0.15, color='#e06c5a', label='AI-driven excess')

ax.axvline(2022.5, color='#888', lw=1, linestyle=':', alpha=0.8)
ax.text(2022.6, ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 0 else 70000,
        'AI acceleration\nbegins', fontsize=8, color='#555')

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_xticks(all_years)
ax.set_xticklabels([str(y) for y in all_years], rotation=45, ha='right')
ax.set_title('Annual CVE Disclosures vs. Pre-AI Trend', fontweight='bold')
ax.set_ylabel('CVEs Published')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# --- Right: YoY % change with CAGR line ---
ax2 = axes[1]
plot_years = df['year'].tolist()
yoy_vals   = df['yoy_pct'].tolist()
bar_colors = ['#4a90d9' if y <= 2022 else '#e06c5a' for y in plot_years]

bars = ax2.bar(plot_years, yoy_vals, color=bar_colors, alpha=0.85, edgecolor='white')
ax2.axhline(cagr, color='black', lw=1.8, linestyle='--',
            label=f'Pre-AI CAGR = {cagr:.1f}%/yr')
ax2.axhline(0, color='#aaa', lw=0.8)

for bar, val in zip(bars, yoy_vals):
    if pd.notna(val):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:+.0f}%', ha='center', va='bottom', fontsize=7.5)

ax2.set_title('Year-over-Year CVE Growth Rate', fontweight='bold')
ax2.set_ylabel('YoY Change (%)')
ax2.set_xticks(plot_years)
ax2.set_xticklabels([str(y) for y in plot_years], rotation=45, ha='right')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

h1 = mpatches.Patch(color='#4a90d9', alpha=0.85, label='Pre-AI era (≤ 2022)')
h2 = mpatches.Patch(color='#e06c5a', alpha=0.85, label='AI-acceleration era (2023+)')
ax2.legend(handles=[h1, h2, ax2.get_lines()[0]], fontsize=8)

fig.suptitle('CVE Discovery: From Predictable Growth to Capability-Triggered Jumps',
             fontsize=13, fontweight='bold')
fig.text(0.5, 0.02,
         'Source: NVD/MITRE published totals. 2025–2026 are estimates. '
         'Note: 2024 counts reflect CVE ID assignments; NVD enrichment lagged '
         'due to the NIST analysis backlog (announced May 2024).',
         ha='center', fontsize=8, color='#666', style='italic',
         transform=fig.transFigure)

plt.savefig(OUTPUT / '01_cve_trends.png', bbox_inches='tight')
plt.show()
print('Chart saved.')

## 5. Quantifying the AI-Era Excess

How many CVEs in 2023 and 2024 are *above* what the pre-AI trend would have predicted?

In [ ]:
print('AI-era structural break analysis:\n')
print(f'{"Year":<6} {"Actual":>10} {"Trend predicts":>16} {"Excess":>10} {"Excess %":>10}')
print('-' * 56)
for year in [2023, 2024]:
    if year not in counts:
        continue
    actual  = counts[year]
    trend   = int(slope * year + intercept)
    excess  = actual - trend
    excess_pct = excess / trend * 100
    print(f'{year:<6} {actual:>10,} {trend:>16,} {excess:>+10,} {excess_pct:>+9.1f}%')

print(f'\nKey takeaway:')
print(f'  The pre-AI trend line (CAGR {cagr:.1f}%/yr) predicts modest linear growth.')
print(f'  The actual data shows step-change jumps tied to AI capability releases.')
print(f'  This is the transition from a time-series model to a capability-triggered model.')

## 6. Try It Yourself

**Experiment:** Change the `BASELINE_END_YEAR` below and re-run to see how the trendline changes. What if the AI era started earlier? Later?

In [ ]:
# --- PARAMETER: change this and re-run ---
BASELINE_END_YEAR = 2022  # try 2021 or 2023

df_base = df[df['year'] <= BASELINE_END_YEAR]
s2, i2, r2, *_ = stats.linregress(df_base['year'], df_base['cves'])
cagr2 = ((df_base['cves'].iloc[-1] / df_base['cves'].iloc[0]) **
          (1 / (len(df_base) - 1)) - 1) * 100

print(f'Baseline end year: {BASELINE_END_YEAR}')
print(f'  Slope:  +{s2:,.0f} CVEs/year')
print(f'  CAGR:   {cagr2:.1f}%/year')
print(f'  R²:     {r2**2:.3f}')
for year in sorted(counts):
    if year > BASELINE_END_YEAR:
        pred = int(s2 * year + i2)
        excess = counts[year] - pred
        print(f'  {year} excess above trend: {excess:+,} ({excess/pred*100:+.1f}%)')